# Cost Analysis for Model Optimization

This notebook analyzes the cost implications of various model optimization techniques we've explored in this workshop, including:

1. Quantization
2. Pruning
3. Knowledge Distillation
4. Fine-tuning

We'll compare the costs of training, inference, and storage for each approach and calculate the potential ROI.

## 1. Import Dependencies and Load Settings

In [ ]:
# Import required libraries
import os
import sys
import time
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Load stored variables from notebook 1
%store -r S3_BUCKET
%store -r SAGEMAKER_ROLE_ARN
%store -r AWS_REGION
%store -r OPTIMIZATION_INSTANCE_TYPE

# Create our own sagemaker session
sagemaker_session = sagemaker.Session()

# Verify variables were loaded
print(f"Loaded variables:")
print(f"S3_BUCKET: {S3_BUCKET}")
print(f"AWS_REGION: {AWS_REGION}")
print(f"OPTIMIZATION_INSTANCE_TYPE: {OPTIMIZATION_INSTANCE_TYPE}")

## 2. Define Cost Models

Let's define the cost structure for different AWS services and instance types used in our optimization techniques.

In [ ]:
# AWS SageMaker pricing (US East - N. Virginia as of 2024)
# Note: Prices may vary by region and change over time

SAGEMAKER_PRICING = {
    # Training instances (per hour)
    'training': {
        'ml.m5.large': 0.115,
        'ml.m5.xlarge': 0.230,
        'ml.m5.2xlarge': 0.461,
        'ml.c5.xlarge': 0.204,
        'ml.c5.2xlarge': 0.408,
        'ml.g4dn.xlarge': 0.736,
        'ml.g4dn.2xlarge': 1.058,
        'ml.g5.xlarge': 1.006,
        'ml.g5.2xlarge': 1.515
    },
    # Processing instances (per hour)
    'processing': {
        'ml.m5.large': 0.115,
        'ml.m5.xlarge': 0.230,
        'ml.c5.xlarge': 0.204,
        'ml.c5.2xlarge': 0.408
    },
    # Inference endpoints (per hour)
    'inference': {
        'ml.t2.medium': 0.065,
        'ml.m5.large': 0.115,
        'ml.m5.xlarge': 0.230,
        'ml.c5.large': 0.102,
        'ml.c5.xlarge': 0.204,
        'ml.g4dn.xlarge': 0.736
    }
}

# S3 storage pricing (per GB per month)
S3_STORAGE_COST = 0.023  # Standard storage

print("Cost models defined successfully")
print(f"Example training cost (ml.g5.xlarge): ${SAGEMAKER_PRICING['training']['ml.g5.xlarge']}/hour")
print(f"S3 storage cost: ${S3_STORAGE_COST}/GB/month")

## 3. Load Optimization Results

Let's load the results from our optimization experiments to analyze their cost implications.

In [ ]:
# Load model information
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

# Create sample optimization results for cost analysis
# In a real scenario, these would be loaded from the actual optimization experiments
optimization_results = {
    "baseline": {
        "model_size_mb": 250,
        "inference_time_ms": 45,
        "accuracy": 0.92,
        "training_time_hours": 0,  # Pre-trained model
        "optimization_time_hours": 0
    },
    "quantized": {
        "model_size_mb": 65,
        "inference_time_ms": 38,
        "accuracy": 0.91,
        "training_time_hours": 0,
        "optimization_time_hours": 0.5
    },
    "pruned": {
        "model_size_mb": 175,
        "inference_time_ms": 32,
        "accuracy": 0.89,
        "training_time_hours": 0,
        "optimization_time_hours": 1.2
    },
    "distilled": {
        "model_size_mb": 85,
        "inference_time_ms": 28,
        "accuracy": 0.90,
        "training_time_hours": 2.5,
        "optimization_time_hours": 2.5
    },
    "fine_tuned": {
        "model_size_mb": 250,
        "inference_time_ms": 45,
        "accuracy": 0.95,
        "training_time_hours": 3.0,
        "optimization_time_hours": 3.0
    }
}

print("Optimization results loaded:")
for technique, results in optimization_results.items():
    print(f"  {technique}: {results['model_size_mb']}MB, {results['inference_time_ms']}ms, {results['accuracy']:.2f} accuracy")

## 4. Calculate Optimization Costs

Let's calculate the one-time costs for each optimization technique.

In [ ]:
def calculate_optimization_costs(results, pricing):
    """
    Calculate the one-time costs for each optimization technique
    """
    costs = {}
    
    for technique, data in results.items():
        if technique == 'baseline':
            costs[technique] = 0  # No optimization cost for baseline
            continue
            
        # Determine instance type based on technique
        if technique in ['fine_tuned', 'distilled']:
            # Training-based techniques use GPU instances
            instance_type = 'ml.g5.xlarge'
            instance_cost = pricing['training'][instance_type]
        else:
            # Processing-based techniques use CPU instances
            instance_type = 'ml.c5.xlarge'
            instance_cost = pricing['processing'][instance_type]
        
        # Calculate total optimization cost
        optimization_hours = data['optimization_time_hours']
        total_cost = optimization_hours * instance_cost
        
        costs[technique] = {
            'instance_type': instance_type,
            'hourly_rate': instance_cost,
            'hours': optimization_hours,
            'total_cost': total_cost
        }
    
    return costs

optimization_costs = calculate_optimization_costs(optimization_results, SAGEMAKER_PRICING)

print("Optimization Costs:")
for technique, cost_data in optimization_costs.items():
    if technique == 'baseline':
        print(f"  {technique}: $0 (no optimization)")
    else:
        print(f"  {technique}: ${cost_data['total_cost']:.2f} ({cost_data['hours']}h @ ${cost_data['hourly_rate']}/h on {cost_data['instance_type']})")

## 5. Summary and Recommendations

Based on our cost analysis, here are the key findings and recommendations:

In [ ]:
print("=" * 80)
print("COST ANALYSIS SUMMARY AND RECOMMENDATIONS")
print("=" * 80)

print("\n1. QUANTIZATION:")
print("   - Pros: Significant model size reduction (74%), low optimization cost")
print("   - Cons: Slight accuracy drop (1%)")
print("   - Best for: Production deployments where model size matters")

print("\n2. PRUNING:")
print("   - Pros: Good inference speed improvement, moderate size reduction")
print("   - Cons: More significant accuracy drop (3%), higher optimization cost")
print("   - Best for: Applications where inference speed is critical")

print("\n3. KNOWLEDGE DISTILLATION:")
print("   - Pros: Best inference speed, good size reduction, balanced accuracy")
print("   - Cons: Highest optimization cost due to training requirements")
print("   - Best for: Long-term deployments with high inference volume")

print("\n4. FINE-TUNING:")
print("   - Pros: Best accuracy improvement (3% gain)")
print("   - Cons: No size/speed benefits, high training cost")
print("   - Best for: Applications where accuracy is paramount")

print("\n" + "=" * 80)
print("GENERAL RECOMMENDATIONS:")
print("=" * 80)

print("\n• For cost-sensitive deployments: Start with Quantization")
print("• For high-volume inference: Consider Knowledge Distillation")
print("• For accuracy-critical applications: Use Fine-tuning")
print("• For real-time applications: Evaluate Pruning vs Distillation")
print("• Consider combining techniques for optimal results")

print("\n" + "=" * 80)
print("NEXT STEPS:")
print("=" * 80)

print("\n1. Run actual experiments with your specific models and data")
print("2. Measure real-world performance metrics")
print("3. Consider your specific cost constraints and requirements")
print("4. Implement monitoring to track ongoing costs and performance")
print("5. Regularly review and optimize based on usage patterns")

print("\n" + "=" * 80)